In [ ]:

!pip install datasets transformers sentencepiece accelerate evaluate -q

import os
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN", "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

print("⏳ Authentification sur HuggingFace...")
login(token=HF_TOKEN)
print("✅ Authentifié sur HuggingFace")

In [ ]:
from datasets import load_dataset, concatenate_datasets, DatasetDict

print("⏳ Chargement des datasets publics Dioula...")

ds_koumankan = load_dataset("uvci/Koumankan_mt_dyu_fr", split="train")
print(f"✅ Koumankan : {len(ds_koumankan)} paires chargées")
print("   Exemple :", ds_koumankan[0])

ds_cv = load_dataset(
    "mozilla-foundation/common_voice_16_1",
    "dyu",
    split="train",
    trust_remote_code=True
)

ds_cv_texte = ds_cv.select_columns(["sentence"])
print(f"✅ Common Voice Dioula : {len(ds_cv_texte)} phrases")

ds_flores = load_dataset("facebook/flores", "dyu_Latn", split="devtest")
print(f"✅ FLORES-200 Dioula : {len(ds_flores)} phrases (évaluation)")

In [ ]:

def formater_traduction(exemple):
    """
    Transforme une paire Dioula-Français en format instruction/réponse
    pour Flan-T5.

    Entrée  : {"dyu": "I ni ce", "fr": "Bonjour"}
    Sortie  : {"input_text": "Traduis en français : I ni ce",
               "target_text": "Bonjour"}
    """
    return {
        "input_text":  f"Traduis en français : {exemple['dyu']}",
        "target_text": exemple["fr"]
    }

def formater_dioula_question(exemple):
    """
    Transforme une phrase Common Voice en paire question/réponse
    pour apprendre au modèle à identifier le dioula.

    Entrée  : {"sentence": "A tun be diɲɛ kɛ"}
    Sortie  : {"input_text": "Quelle langue est cette phrase ? A tun be diɲɛ kɛ",
               "target_text": "Dioula"}
    """
    return {
        "input_text":  f"Quelle langue est cette phrase ? {exemple['sentence']}",
        "target_text": "Dioula"
    }


ds_trad_formate = ds_koumankan.map(formater_traduction, remove_columns=ds_koumankan.column_names)
ds_cv_formate   = ds_cv_texte.map(formater_dioula_question, remove_columns=["sentence"])

print("✅ Koumankan formaté :", ds_trad_formate[0])
print("✅ Common Voice formaté :", ds_cv_formate[0])

ds_complet = concatenate_datasets([ds_trad_formate, ds_cv_formate])
ds_complet = ds_complet.shuffle(seed=42)
print(f"✅ Dataset fusionné : {len(ds_complet)} exemples au total")

In [ ]:

split1 = ds_complet.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

train_ds = split1["train"]
val_ds   = split2["train"]
test_ds  = split2["test"]

print(f"✅ Train      : {len(train_ds)} exemples")
print(f"✅ Validation : {len(val_ds)} exemples")
print(f"✅ Test       : {len(test_ds)} exemples")

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

MODEL_NAME = "google/flan-t5-base"

print(f"⏳ Chargement de {MODEL_NAME}...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
print(f"✅ Modèle chargé — {model.num_parameters():,} paramètres")

def tokeniser(exemples):
    """
    Convertit les textes en tokens numériques compréhensibles par le modèle.

    - max_length=128  : longueur max des phrases d'entrée (coupe au-delà)
    - truncation=True : coupe les phrases trop longues
    - padding="max_length" : complète les phrases trop courtes avec [PAD]

    Les labels = tokens cibles. On remplace le padding (0) par -100
    car CrossEntropyLoss ignore les positions à -100 pendant l'entraînement.
    """
    model_inputs = tokenizer(
        exemples["input_text"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            exemples["target_text"],
            max_length=64,
            truncation=True,
            padding="max_length"
        )

    labels["input_ids"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(tokeniser, batched=True)
val_tok   = val_ds.map(tokeniser, batched=True)
test_tok  = test_ds.map(tokeniser, batched=True)
print("✅ Tokenisation terminée")

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import evaluate
import numpy as np


bleu = evaluate.load("sacrebleu")

def calculer_metriques(eval_pred):
    predictions, labels = eval_pred

    decoded_preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = bleu.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )
    return {"bleu": round(result["score"], 2)}

training_args = Seq2SeqTrainingArguments(
    output_dir                  = "./flan-t5-dioula",
    num_train_epochs            = 5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    logging_dir                 = "./logs",
    logging_steps               = 50,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    predict_with_generate       = True,
    fp16                        = True,
    report_to                   = "none",
)


collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_tok,
    eval_dataset    = val_tok,
    tokenizer       = tokenizer,
    data_collator   = collator,
    compute_metrics = calculer_metriques,
)

print("🚀 Démarrage du fine-tuning...")
trainer.train()
print("✅ Fine-tuning terminé !")

In [ ]:

resultats = trainer.evaluate(test_tok)
print("\n📊 Résultats sur le test set :")
print(f"   Loss BLEU  : {resultats.get('eval_bleu', 'N/A')}")
print(f"   Loss eval  : {resultats.get('eval_loss', 'N/A'):.4f}")


exemples_test = [
    "Traduis en français : I ni ce",
    "Traduis en français : Aw ni baara",
    "Traduis en français : Ne bɛ sɔn",
]

print("\n🔍 Tests manuels :")
for phrase in exemples_test:
    inputs = tokenizer(phrase, return_tensors="pt", max_length=128, truncation=True)
    outputs = model.generate(**inputs, max_length=64, num_beams=4)
    reponse = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"  [{phrase}]  →  {reponse}")

In [ ]:

model.save_pretrained("./flan-t5-dioula-final")
tokenizer.save_pretrained("./flan-t5-dioula-final")
print("✅ Modèle sauvegardé localement dans ./flan-t5-dioula-final")


model.push_to_hub("ton-username/flan-t5-dioula-chatbot")
tokenizer.push_to_hub("ton-username/flan-t5-dioula-chatbot")
print("✅ Modèle publié sur HuggingFace Hub")


!pip install gradio -q

import gradio as gr
from transformers import pipeline


chatbot_pipeline = pipeline(
    "text2text-generation",
    model="./flan-t5-dioula-final",
    tokenizer="./flan-t5-dioula-final"
)

def repondre(message_utilisateur):
    """Fonction appelée à chaque message dans l'interface."""
    prompt  = f"Traduis en français : {message_utilisateur}"
    resultat = chatbot_pipeline(prompt, max_length=64, num_beams=4)
    return resultat[0]["generated_text"]

interface = gr.Interface(
    fn          = repondre,
    inputs      = gr.Textbox(label="Phrase en Dioula"),
    outputs     = gr.Textbox(label="Traduction en Français"),
    title       = "🌍 Chatbot Dioula — TRADITECH 225",
    description = "Traduction Dioula → Français avec Flan-T5 fine-tuné (ESATIC)",
    examples    = [["I ni ce"], ["Aw ni baara"], ["Ne bɛ sɔn k'a sɔrɔ"]]
)

interface.launch(share=True)